# 7.2 多模态端侧部署

视觉语言模型（VLM）、语音模型等多模态模型的端侧部署，需要对不同模态的编码器和解码器分别优化。多模态部署是端侧AI的核心挑战之一，因为不同模态的计算特性差异巨大。

## 什么是多模态端侧部署？

多模态端侧部署是将视觉语言模型（VLM）、语音模型等包含多个模态编码器的模型高效部署到端侧设备的过程。核心挑战是不同模态的编码器计算特性差异大。

## 为什么多模态部署更难？

1. **多编码器**：VLM包含视觉编码器（ViT）和语言模型（LLM），两者计算模式不同
2. **模态间交互**：视觉token和语言token的融合需要额外计算和内存
3. **异构优化**：ViT是compute-bound（大量矩阵乘法），LLM decode是memory-bound（KV Cache读取），需不同优化策略
4. **视觉token压缩**：图像生成的token数量远大于文本，需要压缩以降低LLM的输入长度

## 多模态部署的数学原理

**视觉Token压缩**：将$N$个视觉token压缩为$K$个（$K \ll N$）：
$$z_K = \text{CrossAttn}(q_K, K_V, V_V) = \text{softmax}\left(\frac{q_K K_V^T}{\sqrt{d}}\right) V_V$$
其中$q_K \in \mathbb{R}^{K \times d}$为可学习查询，$K_V, V_V$为视觉特征的Key和Value

**混合精度分配**：视觉编码器和语言模型使用不同量化精度：
$$\text{Size} = \text{Size}_{\text{ViT}}^{\text{INT8}} + \text{Size}_{\text{LLM}}^{\text{INT4}}$$
ViT对量化更鲁棒（INT8几乎无损），LLM decode对精度更敏感（需INT4/GPTQ/AWQ）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### 视觉语言模型（VLM）架构与压缩部署

#### 什么是VLM？

视觉语言模型包含视觉编码器（ViT）和语言模型（LLM），两者计算特性不同，需要分别优化后协同部署。

#### 主流VLM架构

| 模型 | 视觉编码器 | LLM | 融合方式 | 视觉token数 |
|------|-----------|-----|---------|------------|
| LLaVA-1.5 | CLIP-ViT-L/14 (336px) | Vicuna-7B/13B | Linear Projection | 576 |
| LLaVA-NeXT | CLIP-ViT-L/14 (任意分辨率) | Mistral-7B | MLP Projection | 动态 |
| Qwen-VL | ViT-bigG | Qwen-7B | Cross-Attn | 256 |
| InternVL2 | InternViT-6B | InternLM2-8B | MLP | 256 |
| Phi-3-Vision | CLIP-ViT-L | Phi-3-Mini | Linear | 576+ |

#### 为什么VLM部署更难？

1. **双模型架构**：ViT是compute-bound（大量密集矩阵乘法），LLM decode是memory-bound（KV Cache读取）
2. **异构优化需求**：ViT对INT8量化鲁棒（几乎无损），LLM需要INT4/GPTQ量化
3. **跨模态交互**：视觉token和语言token的融合需要额外计算
4. **动态分辨率**：高分辨率图像产生更多token，内存和延迟不可预测

#### VLM压缩的数学原理

混合精度分配：
$$\text{Size} = \text{Size}_{\text{ViT}}^{\text{INT8}} + \text{Size}_{\text{LLM}}^{\text{INT4}}$$

ViT权重$W_{\text{ViT}}$用INT8量化：$s = \frac{\max(|W|)}{127}$
LLM权重$W_{\text{LLM}}$用INT4量化（GPTQ/AWQ）：分组量化，每组独立scale

总压缩比：$\frac{\text{Size}_{\text{FP16}}}{\text{Size}_{\text{compressed}}} = \frac{2(S_{\text{ViT}} + S_{\text{LLM}})}{S_{\text{ViT}}/2 + S_{\text{LLM}}/4}$

典型7B VLM（ViT-L + LLM-7B）：
- ViT-L: ~300M参数，FP16=600MB，INT8=300MB
- LLM-7B: ~7B参数，FP16=14GB，INT4(GPTQ)=3.5GB
- 总计: FP16=14.6GB → 混合量化=3.8GB，压缩比3.8x

In [ ]:
class SimpleViT(nn.Module):
    """简化视觉编码器"""
    def __init__(self, patch_size=16, in_channels=3, dim=128, n_layers=4, n_heads=4):
        super().__init__()
        self.patch_embed = nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=dim, nhead=n_heads, batch_first=True)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        h = self.patch_embed(x)
        h = h.flatten(2).transpose(1, 2)
        for layer in self.layers:
            h = layer(h)
        return self.norm(h)

class VisionLanguageModel(nn.Module):
    """简化VLM"""
    def __init__(self, dim=128, n_layers=4, n_heads=4, vocab_size=1000):
        super().__init__()
        self.vision_encoder = SimpleViT(dim=dim, n_layers=n_layers, n_heads=n_heads)
        self.projection = nn.Linear(dim, dim)
        self.llm = nn.Sequential(
            nn.Embedding(vocab_size, dim),
            nn.TransformerEncoder(
                nn.TransformerEncoderLayer(d_model=dim, nhead=n_heads, batch_first=True),
                num_layers=6
            ),
            nn.Linear(dim, vocab_size)
        )

    def forward(self, image, text_ids):
        vision_features = self.vision_encoder(image)
        projected = self.projection(vision_features)
        text_embed = self.llm[0](text_ids)
        combined = torch.cat([projected, text_embed], dim=1)
        llm_out = self.llm[1](combined)
        logits = self.llm[2](llm_out)
        return logits

vlm = VisionLanguageModel(dim=128, n_layers=4, n_heads=4, vocab_size=1000)

vit_params = sum(p.numel() for p in vlm.vision_encoder.parameters())
proj_params = sum(p.numel() for p in vlm.projection.parameters())
llm_params = sum(p.numel() for p in vlm.llm.parameters())
total_params = vit_params + proj_params + llm_params

print(f"=== VLM参数分布 ===")
print(f"视觉编码器: {vit_params:,} ({vit_params/total_params:.1%})")
print(f"投影层: {proj_params:,} ({proj_params/total_params:.1%})")
print(f"语言模型: {llm_params:,} ({llm_params/total_params:.1%})")
print(f"总计: {total_params:,}")

print(f"\n--- 各组件量化策略 ---")
print(f"视觉编码器(ViT): INT8量化，ViT对量化鲁棒，INT8几乎无损")
print(f"  原因: ViT是compute-bound，INT8可利用INT8 Tensor Core加速")
print(f"  例外: 部分场景可用INT4，但需校准数据集验证精度")
print(f"投影层: 保持FP16，参数量小但影响跨模态对齐质量")
print(f"语言模型(LLM): INT4/GPTQ/AWQ量化，LLM decode是memory-bound")
print(f"  原因: INT4减少内存带宽需求，加速decode阶段")

vit_int8 = vit_params * 1 / (1024 * 1024)
proj_fp16 = proj_params * 2 / (1024 * 1024)
llm_int4 = llm_params * 0.5 / (1024 * 1024)
total_mixed = vit_int8 + proj_fp16 + llm_int4
total_fp32 = total_params * 4 / (1024 * 1024)
total_fp16 = total_params * 2 / (1024 * 1024)

print(f"\n--- 混合量化后存储 ---")
print(f"FP32总存储: {total_fp32:.1f} MB")
print(f"FP16总存储: {total_fp16:.1f} MB")
print(f"混合量化存储: {total_mixed:.1f} MB")
print(f"FP16→混合压缩比: {total_fp16/total_mixed:.1f}x")

### 视觉Token压缩

#### 什么是视觉Token压缩？

图像经ViT编码后产生大量视觉token（如576个），直接输入LLM会导致注意力计算量暴增。视觉Token压缩将$N$个token压缩为$K$个（$K \ll N$）。

#### 为什么需要视觉Token压缩？

1. **计算量降低**：注意力复杂度从$O((N+L)^2)$降至$O((K+L)^2)$
2. **KV Cache减少**：视觉token的KV也需缓存，压缩后缓存减少$N/K$倍
3. **延迟降低**：LLM处理的token数减少，推理速度提升

#### 视觉Token压缩方法对比

| 方法 | 原理 | 压缩比 | 精度损失 | 适用场景 |
|------|------|--------|---------|----------|
| 平均池化 | 相邻token取平均 | 固定 | 中等 | 简单快速 |
| 选择性保留 | 按重要性评分选top-K | 固定 | 低 | 稀疏场景 |
| Cross-Attn池化 | 可学习查询做交叉注意力 | 固定 | 最低 | 精度优先 |
| Token合并(ToMe) | 相似token合并 | 渐进 | 低 | ViT专用 |
| 动态分辨率 | 低分辨率少token | 动态 | 低 | 多尺度输入 |

#### Cross-Attention Token压缩的数学原理

使用可学习查询向量进行注意力池化：
$$z_K = \text{CrossAttn}(q_K, K_V, V_V) = \text{softmax}\left(\frac{q_K K_V^T}{\sqrt{d}}\right) V_V$$

其中：
- $q_K \in \mathbb{R}^{K \times d}$：$K$个可学习查询向量
- $K_V, V_V \in \mathbb{R}^{N \times d}$：视觉特征的Key和Value
- $z_K \in \mathbb{R}^{K \times d}$：压缩后的$K$个视觉token
- 典型配置：$N=576, K=64$，压缩比9:1

Qwen-VL使用此方法将视觉token从256压缩到256（保持不变但通过cross-attn增强），
LLaVA-NeXT通过动态分辨率+patch merging减少token数。

In [ ]:
class CrossAttentionCompressor(nn.Module):
    """Cross-Attention视觉Token压缩器: 使用可学习查询做交叉注意力"""
    def __init__(self, dim, target_tokens=16, n_heads=4):
        super().__init__()
        self.target_tokens = target_tokens
        self.queries = nn.Parameter(torch.randn(1, target_tokens, dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.norm_q = nn.LayerNorm(dim)
        self.norm_v = nn.LayerNorm(dim)

    def forward(self, vision_tokens):
        B = vision_tokens.shape[0]
        queries = self.queries.expand(B, -1, -1)
        q = self.norm_q(queries)
        kv = self.norm_v(vision_tokens)
        compressed, _ = self.cross_attn(q, kv, kv)
        return compressed

class TokenCompressor(nn.Module):
    """选择性Token压缩器: 按重要性评分保留top-K token"""
    def __init__(self, dim, target_tokens=16):
        super().__init__()
        self.target_tokens = target_tokens
        self.merge_weights = nn.Linear(dim, 1)

    def forward(self, vision_tokens):
        B, N, D = vision_tokens.shape
        if N <= self.target_tokens:
            return vision_tokens
        importance = self.merge_weights(vision_tokens).squeeze(-1)
        importance = F.softmax(importance, dim=-1)
        topk_indices = importance.topk(self.target_tokens, dim=-1).indices
        topk_indices = topk_indices.sort(dim=-1).values
        selected = torch.gather(vision_tokens, 1, topk_indices.unsqueeze(-1).expand(-1, -1, D))
        return selected

class AveragePoolingCompressor(nn.Module):
    """平均池化压缩器"""
    def __init__(self, target_tokens=16):
        super().__init__()
        self.target_tokens = target_tokens

    def forward(self, vision_tokens):
        B, N, D = vision_tokens.shape
        if N <= self.target_tokens:
            return vision_tokens
        group_size = N // self.target_tokens
        trimmed = vision_tokens[:, :group_size * self.target_tokens, :]
        pooled = trimmed.reshape(B, self.target_tokens, group_size, D).mean(dim=2)
        return pooled

dim = 128
n_original_tokens = 196
target_tokens = 16
tokens = torch.randn(2, n_original_tokens, dim)

compressor_cross = CrossAttentionCompressor(dim, target_tokens, n_heads=4)
compressor_select = TokenCompressor(dim, target_tokens)
compressor_pool = AveragePoolingCompressor(target_tokens)

with torch.no_grad():
    compressed_cross = compressor_cross(tokens)
    compressed_select = compressor_select(tokens)
    compressed_pool = compressor_pool(tokens)

print(f"=== 视觉Token压缩方法对比 ===")
print(f"原始token数: {n_original_tokens}")
print(f"目标token数: {target_tokens}")
print(f"压缩比: {n_original_tokens/target_tokens:.1f}x")
print(f"\n{'方法':<20} {'压缩后shape':<20} {'可学习参数':<15} {'特点':<20}")
print("-" * 75)
cross_params = sum(p.numel() for p in compressor_cross.parameters())
select_params = sum(p.numel() for p in compressor_select.parameters())
print(f"{'Cross-Attention':<20} {str(list(compressed_cross.shape)):<20} {cross_params:<15,} {'精度最高':<20}")
print(f"{'选择性保留':<20} {str(list(compressed_select.shape)):<20} {select_params:<15,} {'稀疏友好':<20}")
print(f"{'平均池化':<20} {str(list(compressed_pool.shape)):<20} {'0':<15} {'最简单':<20}")

print(f"\n--- 对LLM推理的影响 ---")
original_attn_ops = n_original_tokens ** 2
compressed_attn_ops = target_tokens ** 2
print(f"  自注意力计算量: {original_attn_ops} -> {compressed_attn_ops} ({compressed_attn_ops/original_attn_ops*100:.1f}%)")
print(f"  KV Cache: {n_original_tokens} -> {target_tokens} tokens")
print(f"  内存节省: {(1 - target_tokens/n_original_tokens)*100:.0f}%")

print(f"\n--- 产业界视觉Token压缩实践 ---")
print(f"1. Qwen-VL: Cross-Attention压缩，256个视觉token")
print(f"2. LLaVA-NeXT: 动态分辨率+patch merging")
print(f"3. InternVL2: 256个压缩token，6B ViT编码器")
print(f"4. ToMe(Token Merging): 训练时渐进合并相似token，无需额外参数")

### 语音端侧部署

#### 什么是语音端侧部署？

语音模型（ASR/TTS）的端侧部署，核心挑战是流式处理和低延迟要求。语音交互要求端到端延迟<300ms（用户感知阈值）。

#### 语音模型的特殊挑战

1. **流式处理**：语音是连续信号，需要逐帧处理而非整段处理
2. **低延迟要求**：实时对话要求首token延迟<200ms
3. **长序列**：30秒音频经特征提取后可能产生数百帧
4. **多阶段pipeline**：ASR通常包含特征提取→编码→解码→文本生成

#### 主流端侧语音架构

| 模型 | 类型 | 参数量 | 延迟 | 适用场景 |
|------|------|--------|------|----------|
| Whisper-small | ASR | 244M | ~500ms | 离线转写 |
| Whisper-tiny | ASR | 39M | ~100ms | 端侧实时 |
| Paraformer | ASR | 220M | ~50ms | 流式识别 |
| Sherpa-ONNX | ASR | 可变 | <100ms | 嵌入式设备 |
| VITS | TTS | 50-100M | ~100ms | 端侧合成 |
| CosyVoice | TTS | 200M+ | ~200ms | 高质量合成 |

#### 语音端侧优化策略

- **流式架构**：chunked attention，每次处理固定长度音频帧
- **INT8量化**：编码器和解码器均可INT8量化，精度损失小
- **ONNX导出**：使用Sherpa-ONNX等框架在嵌入式设备上运行
- **特征缓存**：缓存已处理的音频特征，避免重复计算

In [ ]:
class StreamingAudioEncoder(nn.Module):
    """流式音频编码器: 逐chunk处理音频特征"""
    def __init__(self, feat_dim=80, d_model=256, n_layers=4, n_heads=4, chunk_size=32):
        super().__init__()
        self.chunk_size = chunk_size
        self.embed = nn.Linear(feat_dim, d_model)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, batch_first=True)
            for _ in range(n_layers)
        ])
        self.cache = None

    def reset_cache(self):
        self.cache = None

    def forward_chunk(self, chunk):
        """处理一个音频chunk"""
        h = self.embed(chunk)
        if self.cache is not None:
            h = torch.cat([self.cache, h], dim=1)
        for layer in self.layers:
            h = layer(h)
        self.cache = h[:, -self.chunk_size:, :].detach()
        return h

encoder = StreamingAudioEncoder(feat_dim=80, d_model=256, n_layers=4, n_heads=4, chunk_size=32)
encoder.eval()

n_chunks = 5
chunk_size = 32
feat_dim = 80

print(f"=== 流式音频编码器 ===")
print(f"特征维度: {feat_dim}")
print(f"模型维度: 256")
print(f"chunk大小: {chunk_size}帧")
print(f"\n流式处理模拟:")

encoder.reset_cache()
total_frames = 0
for i in range(n_chunks):
    chunk = torch.randn(1, chunk_size, feat_dim)
    with torch.no_grad():
        output = encoder.forward_chunk(chunk)
    total_frames += chunk_size
    print(f"  Chunk {i+1}: 输入{chunk_size}帧, 输出shape={list(output.shape)}, 缓存={chunk_size}帧")

print(f"\n总处理帧数: {total_frames}")
print(f"流式优势: 逐chunk处理，无需等待完整音频，降低首token延迟")

print(f"\n=== 多模态端侧部署总结 ===")
print(f"\n{'模态':<10} {'核心挑战':<20} {'关键优化':<25} {'典型模型':<20}")
print("-" * 75)
modalities = [
    ('视觉', 'token数量大', 'Token压缩+INT8量化', 'LLaVA/Qwen-VL'),
    ('语音', '流式+低延迟', 'chunked attn+INT8', 'Whisper/Paraformer'),
    ('多模态', '异构计算模式', '混合精度+分阶段优化', 'InternVL2'),
]
for mod, challenge, optimization, model in modalities:
    print(f"{mod:<10} {challenge:<20} {optimization:<25} {model:<20}")

---
### ViT编码器的端侧优化

视觉编码器(ViT)是VLM中计算量最大的组件，在端侧需要针对性优化。

#### ViT计算瓶颈分析

ViT的计算主要由三部分组成：

| 组件 | 计算量(FLOPs) | 访存量 | 瓶颈类型 |
|------|-------------|--------|----------|
| **Patch Embedding** | $2hw \cdot p^2 \cdot 3 \cdot d$ | $hwp^2 \cdot 3$ | compute-bound |
| **Self-Attention** | $2N^2 d + 4Nd^2$ | $2N^2 + 4Nd$ | compute-bound(N<1024) |
| **FFN** | $8Nd^2$ | $16Nd$ | compute-bound |

#### Token Pruning：动态丢弃不重要patch

ViT的注意力分数揭示了不同patch的重要性：背景patch的注意力分数低，前景patch分数高。

$$\text{importance}(p_i) = \frac{1}{L} \sum_{l=1}^{L} \text{Attn}_{[CLS], p_i}^{(l)}$$

保留Top-K个重要token，丢弃其余：

$$\text{Tokens}_{\text{keep}} = \text{TopK}(\text{importance}, K = \lfloor \rho \cdot N \rfloor)$$

其中$\rho$为保留比例（如0.7=保留70%token）。

#### ViT量化特性

ViT对量化比LLM更鲁棒：
- **INT8几乎无损**：ViT的激活分布较均匀，INT8量化PPL增加<0.5%
- **INT4可行**：权重INT4+激活INT8的混合精度，PPL增加<2%
- **LayerNorm优于BatchNorm**：LayerNorm对量化噪声更鲁棒

In [ ]:
class ViTOptimizer:
    """ViT编码器端侧优化模拟"""
    def __init__(self, img_size=224, patch_size=16, embed_dim=1024, depth=24, n_heads=16):
        self.img_size = img_size
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.depth = depth
        self.n_heads = n_heads
        self.n_patches = (img_size // patch_size) ** 2

    def compute_flops(self, n_tokens=None):
        """计算FLOPs"""
        N = n_tokens or self.n_patches
        d = self.embed_dim
        patch_emb_flops = 2 * self.img_size * self.img_size * self.patch_size * self.patch_size * 3 * d
        attn_flops = self.depth * (2 * N * N * d + 4 * N * d * d)
        ffn_flops = self.depth * 8 * N * d * d
        total = patch_emb_flops + attn_flops + ffn_flops
        return {'patch_emb': patch_emb_flops, 'attn': attn_flops, 'ffn': ffn_flops, 'total': total}

    def simulate_token_pruning(self, keep_ratios=[1.0, 0.8, 0.6, 0.4]):
        """模拟Token Pruning效果"""
        results = []
        for ratio in keep_ratios:
            n_keep = int(self.n_patches * ratio)
            flops = self.compute_flops(n_keep)
            base_flops = self.compute_flops(self.n_patches)
            results.append({
                'keep_ratio': ratio,
                'n_tokens': n_keep,
                'total_gflops': flops['total'] / 1e9,
                'flops_reduction': 1 - flops['total'] / base_flops['total'],
                'accuracy_retention': 1.0 - 0.02 * (1 - ratio) * self.depth,
            })
        return results

vit = ViTOptimizer(img_size=336, patch_size=14, embed_dim=1024, depth=24)
print(f"=== ViT编码器分析 (ViT-L/14@336px, CLIP视觉编码器) ===")
print(f"Patch数: {vit.n_patches}")

flops = vit.compute_flops()
print(f"\n--- 计算量分解 ---")
print(f"  Patch Embedding: {flops['patch_emb']/1e9:.1f} GFLOPs ({flops['patch_emb']/flops['total']*100:.1f}%)")
print(f"  Self-Attention:  {flops['attn']/1e9:.1f} GFLOPs ({flops['attn']/flops['total']*100:.1f}%)")
print(f"  FFN:             {flops['ffn']/1e9:.1f} GFLOPs ({flops['ffn']/flops['total']*100:.1f}%)")
print(f"  总计:            {flops['total']/1e9:.1f} GFLOPs")

print(f"\n--- Token Pruning效果 ---")
print(f"{'保留比例':<10} {'Token数':<10} {'GFLOPs':<10} {'计算减少':<10} {'精度保留'}")
print("-" * 50)
for r in vit.simulate_token_pruning():
    print(f"{r['keep_ratio']:<10.0%} {r['n_tokens']:<10} {r['total_gflops']:<10.1f} "
          f"{r['flops_reduction']:<10.1%} {r['accuracy_retention']:.1%}")

print(f"\n--- ViT量化效果 ---")
quant_results = [
    ('FP16', 2, 0.0),
    ('INT8(W8A8)', 1, 0.3),
    ('INT4(W4A8)', 0.5, 1.5),
]
model_size_mb = 24 * 1024 * 1024 * 1024 * 2 / 1e6
print(f"{'量化方案':<15} {'模型大小(MB)':<15} {'PPL增加':<10}")
for name, bw, ppl_inc in quant_results:
    size = model_size_mb * bw / 2
    print(f"{name:<15} {size:<15.0f} {ppl_inc}%")

print(f"\n=== 产业实践要点 ===")
print(f"1. ViT的INT8量化几乎无损，应优先使用")
print(f"2. Token Pruning在背景简单的图像上效果好，复杂场景需谨慎")
print(f"3. Patch Embedding可用Conv2d→Im2Col+GEMM优化，适配NPU")
print(f"4. MobileViT/EfficientViT是端侧首选架构，FLOPs仅ViT-L的1/10")
print(f"5. ViT的Flash Attention适配比LLM简单（序列长度短）")

---
### 多模态推理调度

VLM推理分为视觉编码和语言解码两个阶段，两者的计算模式完全不同，需要异构调度。

#### 两阶段计算模式

| 阶段 | 计算 | 瓶颈 | 适合硬件 |
|------|------|------|----------|
| **视觉编码(Prefill)** | ViT前向+投影 | compute-bound | NPU/GPU |
| **语言解码(Decode)** | LLM逐token生成 | memory-bound | NPU(低功耗)/CPU |

#### 流水线调度

```
时间 →
NPU:  [ViT编码] [LLM Decode Layer 0-15] [LLM Decode Layer 16-31]
CPU:            [LLM Decode Layer 16-31]  [LLM Decode Layer 0-15]
```

当NPU完成ViT编码后，NPU和CPU可以并行处理LLM的不同层。

#### 批处理策略

- **视觉请求批处理**：多张图像的ViT编码可以batch处理（compute-bound）
- **语言请求批处理**：多请求的LLM decode可连续批处理（memory-bound）
- **混合批处理**：视觉和语言请求不能混合（计算模式不同）

In [ ]:
class MultimodalScheduler:
    """多模态推理调度器"""
    def __init__(self):
        self.devices = {
            'NPU': {'compute_tops': 75, 'memory_bw_gbs': 60, 'power_w': 5},
            'CPU': {'compute_tops': 2, 'memory_bw_gbs': 40, 'power_w': 3},
            'GPU': {'compute_tops': 200, 'memory_bw_gbs': 100, 'power_w': 15},
        }

    def schedule_vlm_inference(self, vit_gflops, llm_params_b, llm_dtype_bytes=0.5, n_decode_tokens=100):
        """调度VLM推理"""
        vit_time_npu = vit_gflops / self.devices['NPU']['compute_tops'] * 1000
        vit_time_cpu = vit_gflops / self.devices['CPU']['compute_tops'] * 1000
        vit_time_gpu = vit_gflops / self.devices['GPU']['compute_tops'] * 1000

        decode_bytes_per_token = llm_params_b * 1e9 * llm_dtype_bytes
        decode_time_npu = decode_bytes_per_token / (self.devices['NPU']['memory_bw_gbs'] * 1e9) * 1000
        decode_time_cpu = decode_bytes_per_token / (self.devices['CPU']['memory_bw_gbs'] * 1e9) * 1000

        schedules = {
            'NPU全做': {
                'vit_device': 'NPU', 'llm_device': 'NPU',
                'vit_ms': vit_time_npu, 'decode_ms_per_tok': decode_time_npu,
                'total_ms': vit_time_npu + decode_time_npu * n_decode_tokens,
                'power_w': self.devices['NPU']['power_w'],
            },
            'NPU(ViT)+CPU(LLM)': {
                'vit_device': 'NPU', 'llm_device': 'CPU',
                'vit_ms': vit_time_npu, 'decode_ms_per_tok': decode_time_cpu,
                'total_ms': vit_time_npu + decode_time_cpu * n_decode_tokens,
                'power_w': self.devices['NPU']['power_w'] + self.devices['CPU']['power_w'],
            },
            'GPU(ViT)+NPU(LLM)': {
                'vit_device': 'GPU', 'llm_device': 'NPU',
                'vit_ms': vit_time_gpu, 'decode_ms_per_tok': decode_time_npu,
                'total_ms': vit_time_gpu + decode_time_npu * n_decode_tokens,
                'power_w': self.devices['GPU']['power_w'] + self.devices['NPU']['power_w'],
            },
        }
        return schedules

sched = MultimodalScheduler()
results = sched.schedule_vlm_inference(vit_gflops=50, llm_params_b=1.5, n_decode_tokens=100)

print("=== VLM推理调度对比 (1.5B LLM + ViT-L) ===")
print(f"\n{'调度方案':<22} {'ViT设备':<8} {'LLM设备':<8} {'ViT(ms)':<10} {'Decode(ms/tok)':<16} {'总延迟(ms)':<12} {'功耗(W)'}")
print("-" * 90)
for name, r in results.items():
    print(f"{name:<22} {r['vit_device']:<8} {r['llm_device']:<8} {r['vit_ms']:<10.1f} "
          f"{r['decode_ms_per_tok']:<16.1f} {r['total_ms']:<12.0f} {r['power_w']}")

print(f"\n=== 产业实践要点 ===")
print(f"1. ViT编码是compute-bound，应分配给NPU/GPU")
print(f"2. LLM decode是memory-bound，NPU和CPU差异不大")
print(f"3. NPU(ViT)+CPU(LLM)是常见方案: ViT快+LLM省电")
print(f"4. 长序列decode时，LLM延迟主导，ViT编码可忽略")
print(f"5. 多请求场景: ViT可batch处理，LLM用连续批处理")

---
### 端侧VLM部署实战

#### LLaVA-1.5在骁龙NPU上的部署

```
原始模型 (LLaVA-1.5-7B)
  │
  ├── ViT-L/14 (300M params)
  │   ├── INT8量化 → 300MB
  │   └── QNN编译 → NPU执行
  │
  ├── 投影层 (2M params)
  │   └── FP16 → CPU执行
  │
  └── LLM (Vicuna-7B)
      ├── INT4量化(AWQ) → 3.5GB
      └── QNN编译 → NPU执行

总内存: ~3.8GB
```

#### 延迟分解

| 阶段 | 延迟 | 占比 | 优化手段 |
|------|------|------|----------|
| **图像预处理** | 5ms | 1% | NEON指令加速 |
| **ViT编码** | 80ms | 15% | INT8+算子融合 |
| **投影+Token压缩** | 10ms | 2% | 融合到ViT最后一层 |
| **LLM Prefill** | 150ms | 28% | Flash Attention |
| **LLM Decode** | 2700ms(100tok) | 54% | INT4+KV Cache优化 |

#### 关键优化技术

1. **视觉Token压缩**：576个ViT token→64个LLM token，减少89%的LLM输入
2. **混合精度**：ViT INT8 + LLM INT4，总内存<4GB
3. **算子融合**：ViT最后一层+投影+Token压缩融合为单个kernel
4. **KV Cache优化**：PagedAttention + INT8 KV Cache

In [ ]:
class VLMDeploymentSimulator:
    """VLM端侧部署模拟器"""
    def __init__(self, vit_params_m=300, llm_params_b=7, n_visual_tokens=576, n_compressed_tokens=64):
        self.vit_params_m = vit_params_m
        self.llm_params_b = llm_params_b
        self.n_visual_tokens = n_visual_tokens
        self.n_compressed_tokens = n_compressed_tokens

    def simulate_deployment(self, npu_tops=75, dram_bw_gbs=60, n_decode_tokens=100):
        """模拟VLM部署性能"""
        vit_size_mb = self.vit_params_m * 1e6 * 1 / 1e6
        llm_size_mb = self.llm_params_b * 1e9 * 0.5 / 1e6
        total_size_mb = vit_size_mb + llm_size_mb

        vit_flops = self.vit_params_m * 1e6 * 6 * 24
        vit_time_ms = vit_flops / (npu_tops * 1e12 / 1e3) * 1000

        total_input_tokens = self.n_compressed_tokens + 20
        llm_prefill_flops = 2 * total_input_tokens * self.llm_params_b * 1e9
        llm_prefill_time_ms = llm_prefill_flops / (npu_tops * 1e12 / 1e3) * 1000

        decode_bytes = self.llm_params_b * 1e9 * 0.5
        decode_time_per_tok_ms = decode_bytes / (dram_bw_gbs * 1e9 / 1e3) * 1000
        llm_decode_time_ms = decode_time_per_tok_ms * n_decode_tokens

        total_time_ms = vit_time_ms + llm_prefill_time_ms + llm_decode_time_ms

        return {
            'vit_size_mb': vit_size_mb, 'llm_size_mb': llm_size_mb, 'total_size_mb': total_size_mb,
            'vit_time_ms': vit_time_ms, 'llm_prefill_time_ms': llm_prefill_time_ms,
            'decode_time_per_tok_ms': decode_time_per_tok_ms, 'llm_decode_time_ms': llm_decode_time_ms,
            'total_time_ms': total_time_ms,
            'token_compression_ratio': 1 - self.n_compressed_tokens / self.n_visual_tokens,
        }

vlm = VLMDeploymentSimulator()
result = vlm.simulate_deployment()

print("=== VLM端侧部署性能分析 (LLaVA-1.5-7B) ===")
print(f"\n--- 模型大小 ---")
print(f"  ViT (INT8): {result['vit_size_mb']:.0f} MB")
print(f"  LLM (INT4): {result['llm_size_mb']:.0f} MB")
print(f"  总计:       {result['total_size_mb']:.0f} MB")
print(f"\n--- 延迟分解 ---")
print(f"  ViT编码:      {result['vit_time_ms']:.1f} ms ({result['vit_time_ms']/result['total_time_ms']*100:.1f}%)")
print(f"  LLM Prefill:  {result['llm_prefill_time_ms']:.1f} ms ({result['llm_prefill_time_ms']/result['total_time_ms']*100:.1f}%)")
print(f"  LLM Decode:   {result['llm_decode_time_ms']:.1f} ms ({result['llm_decode_time_ms']/result['total_time_ms']*100:.1f}%)")
print(f"  总延迟:       {result['total_time_ms']:.0f} ms")
print(f"\n--- Token压缩效果 ---")
print(f"  原始视觉Token: {vlm.n_visual_tokens}")
print(f"  压缩后Token:   {vlm.n_compressed_tokens}")
print(f"  压缩率:        {result['token_compression_ratio']:.0%}")

print(f"\n=== 产业实践要点 ===")
print(f"1. LLM Decode是延迟主导(>50%)，优化重点在KV Cache和权重读取")
print(f"2. 视觉Token压缩是关键: 576→64可减少89%的LLM prefill计算")
print(f"3. 混合精度(ViT INT8 + LLM INT4)是端侧VLM的标准方案")
print(f"4. 算子融合: ViT最后一层+投影+压缩融合，减少中间结果写回")
print(f"5. 首token延迟(TTFT) = ViT + Prefill，约200-300ms")
print(f"6. 用户感知: TTFT<500ms可接受，Decode>10 tok/s流畅")

---
## 视频理解端侧部署

视频理解是多模态部署的高频产业需求，相比图像理解，视频增加了时序维度，对计算和内存的需求成倍增长。

### 视频理解的关键挑战

| 挑战 | 图像理解 | 视频理解 | 增长倍数 |
|------|---------|---------|----------|
| **输入Token数** | 576 (24×24) | 576×N帧 | Nx |
| **编码器FLOPs** | ~50 GFLOPs | ~50N GFLOPs | Nx |
| **内存占用** | ~1MB/帧 | ~N MB | Nx |
| **延迟要求** | <500ms TTFT | <1s 首帧 | 更宽松但更复杂 |

### 视频帧采样策略

| 策略 | 帧数 | 信息保留 | 计算量 | 适用场景 |
|------|------|---------|--------|----------|
| **均匀采样** | 8-16帧 | 中 | 中 | 通用视频理解 |
| **关键帧采样** | 4-8帧 | 高(场景变化处) | 低 | 长视频摘要 |
| **场景检测采样** | 自适应 | 最高 | 低(检测)+高(编码) | 精确理解 |
| **时序聚合** | 1帧(特征聚合) | 中 | 最低 | 实时流处理 |

### 产业实践要点

1. **帧数控制**：端侧视频理解通常限制在4-8帧，超过8帧内存和延迟难以接受
2. **时序建模轻量化**：用Temporal Pooling替代时序Attention，减少计算量
3. **流式处理**：实时视频流采用滑动窗口+增量编码，避免全量重编码
4. **ViT复用**：视频帧共享同一个ViT编码器，batch处理提升MAC利用率

---
## 总结与最佳实践

### 多模态端侧部署的核心原则

1. **视觉Token压缩是第一优化**：576→64可减少89%的LLM prefill计算，ROI最高
2. **混合精度是标配**：ViT INT8 + LLM INT4，平衡精度与性能
3. **异构调度是关键**：ViT→NPU/GPU, LLM→NPU/CPU，根据硬件能力分配
4. **TTFT是用户体验指标**：ViT+Prefill<500ms，Decode>10 tok/s

### 多模态部署Checklist

- [ ] 选择视觉Token压缩策略（平均池化/注意力合并/学习型压缩）
- [ ] 确定混合精度配置（ViT INT8 + LLM INT4）
- [ ] 实现异构推理调度（ViT和LLM分配到不同计算单元）
- [ ] 处理NPU不支持的算子（自定义算子/CPU fallback）
- [ ] 视频场景：确定帧采样策略和时序建模方案
- [ ] 语音场景：实现流式编码和增量推理
- [ ] 端到端延迟测试：TTFT、Decode速度、内存峰值
- [ ] 多模态内容安全过滤（图像/语音/文本审核）